<a href="https://colab.research.google.com/github/bidallei/MIAAD-UACJ/blob/main/Practica_CBR__SigueTuSalud_AlvaroHernandez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* Asignatura: Sistemas de recomendación
* Profesor: Dra. Karla Olmos Sánchez
* Alumno: **Álvaro Hernández Jarquín**
* Matrícula: 263150
* Fecha de entrega: 2 de noviembre de 2026




## Actividad integradora: aplicar CBR a su proyecto

Responde con argumentos y ejemplos de tu dominio:

1. **Representación del conocimiento.**

¿Qué es un caso? Distingue problema, solución y resultado.

Para mi propuesta que se refiere a una aplicación que recomienda procedimientos de salud a pacientes de acuerdo a sus registros, el problema sería el perfil clínico y logístico de la paciente al momento de usar la app: edad, síntomas (intensidad de dolor, periodo), medicamentos que usa, antecedentes de riesgo, zona de residencia e historial (derivaciones, centro previo); la solución sería la recomendación de procedimiento, por ejemplo, "Urgencias ginecológicas - Hospital B" o "Mastografía preventiva - Clínica C"; y el resultado sería la retroalimentación explícita, que es la valoración que da la paciente al servicio y la implícita, que es si acudió a la cita o si rechazó la recomendación, y los datos capturados por el médico.

2. **Naturaleza del dominio.**

¿Es estructurado o de estructura informal? ¿Qué experiencias serían difíciles de formalizar?

Considero que tiene de ambos dominios. Es muy estructurado en la parte administrativa y edad, tiempos de espera, disponibilidad de equipo, zona, pero la percepción de la salud tiene una estructura informal. Es difícil formalizar la descripción de un síntoma, el dolor de cada paciente y la falta de conocimiento de los nombres de los síntomas o qué categorizar como síntoma en contraposición contra algo normal.

3. **Conocimiento explícito y tácito.**

¿Qué proviene de registros y qué requeriría especialistas o usuarios?

El conocimiento explícito son el catálogo de servicios (NOM-004-SSA3-2012), la ubicación de los Centros de Salud y si hay mastógrafo disponible y el tácito es un médico que sabe interpretar los síntomas aparentemente aislados que son una urgencia silenciosa. Se requiere expertos médicos que validen los pesos y las reglas del algoritmo. En cuanto al usuario, el conocimiento tácito es su percepción de sus síntomas.

4. **Calidad y ética.**

¿Qué riesgos hay al reutilizar casos sesgados, incorrectos o desactualizados?

En este caso, dado que se trabaja con la salud de pacientes, los riesgos son muy altos y responsabilidad legal también. Si el sistema retiene un caso donde una paciente con dolor agudo fue recomendada a "Orientación virtual" y sobrevivió por azar, reutilizar su caso sería una negligencia médica para la siguiente paciente. En cuanto a los sesgos, si hay información histórica de que las mujeres de la zona "Norte" rechazan consultas especializadas por lejanía, el algoritmo podría aprender a dejar de recomendar especialidades a esa zona, lo cuál sería muy peligroso.

5. **Pertinencia.**

¿Por qué utilizarías o descartarías CBR frente a contenido, filtrado colaborativo y reglas?

En SigueTuSalud, un enfoque de Reglas Híbridas (Reglas + Filtrado/CBR) sería lo ideal. Descarto usar sólo Filtrado Colaborativo o CBR porque la salud es un asunto muy serio. La viabilidad clínica. Utilizaría CBR o Filtrado Colaborativo en la etapa final de jerarquización para humanizar y optimizar la recomendación. Una vez que las Reglas garantizaron que las opciones son médicamente seguras, el CBR ayuda a priorizar basándose en el historial de éxito de pacientes similares.


In [1]:
import pandas as pd
from pprint import pprint

# 1. Catálogo. Soluciones posibles
SERVICIOS = [
    dict(id='S1', nombre='Consulta medicina familiar - Centro A', tipo='presencial',
         especialidad='general', tiempo_espera_dias=1, requiere_derivacion=False,
         tiene_equipo_estudios=False, edad_minima=18, zona='Norte', centro='Centro A'),

    dict(id='S2', nombre='Ginecología - Hospital B', tipo='presencial',
         especialidad='especialidad', tiempo_espera_dias=5, requiere_derivacion=True,
         tiene_equipo_estudios=True, edad_minima=18, zona='Sur', centro='Hospital B'),

    dict(id='S3', nombre='Mastografía preventiva - Clínica C', tipo='estudio',
         especialidad='imagenologia', tiempo_espera_dias=2, requiere_derivacion=False,
         tiene_equipo_estudios=True, edad_minima=40, zona='Norte', centro='Clínica C'),

    dict(id='S4', nombre='Orientación médica virtual', tipo='virtual',
         especialidad='general', tiempo_espera_dias=0, requiere_derivacion=False,
         tiene_equipo_estudios=False, edad_minima=18, zona='Todas', centro='Digital'),

    dict(id='S5', nombre='Urgencias ginecológicas - Hospital B', tipo='presencial',
         especialidad='especialidad', tiempo_espera_dias=0, requiere_derivacion=False,
         tiene_equipo_estudios=True, edad_minima=18, zona='Sur', centro='Hospital B')
]

In [2]:
# 2. Usuarios actuales
PACIENTE_1 = dict(
    nombre='Pancracia', edad=32, urgencia_dias=1, tiene_derivacion=False,
    requiere_equipo=False, zona_residencia='Norte', centro_previo='Centro A',
    antecedentes_riesgo=False, dolor_intensidad=8
)

PACIENTE_2 = dict(
    nombre='Ricarda', edad=45, urgencia_dias=15, tiene_derivacion=True,
    requiere_equipo=True, zona_residencia='Sur', centro_previo='Hospital B',
    antecedentes_riesgo=True, dolor_intensidad=2
)

In [3]:
# 3. Filtros
def evaluar_restricciones_salud(servicio, paciente):
    razones = []
    # R1: Urgencia vs Tiempo de espera
    if servicio['tiempo_espera_dias'] > paciente['urgencia_dias']:
        razones.append('R1: Tiempo de espera supera la tolerancia clínica')
    # R2: Requisito de derivación
    if servicio['requiere_derivacion'] and not paciente['tiene_derivacion']:
        razones.append('R2: Requiere pase a especialidad')
    # R3: Capacidad técnica
    if paciente['requiere_equipo'] and not servicio['tiene_equipo_estudios']:
        razones.append('R3: El centro no cuenta con el equipo requerido')
    # R4: Criterio de edad
    if paciente['edad'] < servicio['edad_minima']:
        razones.append('R4: Paciente fuera del rango de edad para este protocolo')
    return razones

In [4]:
# 4. Similitud
def calcular_puntuacion_salud(servicio, paciente):
    puntos = 0
    explicaciones = []
    # P1: Proximidad
    if servicio['zona'] == paciente['zona_residencia'] or servicio['zona'] == 'Todas':
        puntos += 20
        explicaciones.append('P1: Ubicación conveniente (+20)')
    # P2: Continuidad
    if servicio['centro'] == paciente['centro_previo']:
        puntos += 15
        explicaciones.append('P2: Continuidad de atención en mismo centro (+15)')
    # P3: Especialización por antecedentes
    if paciente['antecedentes_riesgo'] and servicio['especialidad'] == 'especialidad':
        puntos += 25
        explicaciones.append('P3: Especialista adecuado para antecedentes (+25)')
    # P4: Modalidad por severidad
    if paciente['dolor_intensidad'] >= 7 and servicio['tipo'] == 'presencial':
        puntos += 30
        explicaciones.append('P4: Atención presencial priorizada por dolor (+30)')
    return puntos, explicaciones

In [5]:
# 5. Recomendación
def recomendar_salud(paciente, servicios, n=3):
    recomendaciones = []
    descartadas = []

    for servicio in servicios:
        # Primero revisamos viabilidad
        razones = evaluar_restricciones_salud(servicio, paciente)
        if razones:
            descartadas.append({'nombre': servicio['nombre'], 'razones': razones})
        else:
            # Si es viable, calculamos su similitud/pertinencia
            puntos, explicaciones = calcular_puntuacion_salud(servicio, paciente)
            recomendaciones.append({'nombre': servicio['nombre'], 'puntos': puntos, 'explicaciones': explicaciones})

    # Ordenamos de mayor a menor puntuación
    recomendaciones.sort(key=lambda r: r['puntos'], reverse=True)

    # Retornamos el Top-N y las opciones descartadas
    return recomendaciones[:n], descartadas


In [6]:
for paciente in [PACIENTE_1, PACIENTE_2]:
    recomendaciones, descartadas = recomendar_salud(paciente, SERVICIOS)

    print('\n' + '='*40)
    print(f"PACIENTE: {paciente['nombre']}")
    print('='*40)

    print('\n✅ RECOMENDACIONES (Top-N):')
    pprint(recomendaciones, sort_dicts=False)

    print('\n❌ DESCARTADAS (Por restricciones clínicas/logísticas):')
    pprint(descartadas, sort_dicts=False)


PACIENTE: Pancracia

✅ RECOMENDACIONES (Top-N):
[{'nombre': 'Consulta medicina familiar - Centro A',
  'puntos': 65,
  'explicaciones': ['P1: Ubicación conveniente (+20)',
                    'P2: Continuidad de atención en mismo centro (+15)',
                    'P4: Atención presencial priorizada por dolor (+30)']},
 {'nombre': 'Urgencias ginecológicas - Hospital B',
  'puntos': 30,
  'explicaciones': ['P4: Atención presencial priorizada por dolor (+30)']},
 {'nombre': 'Orientación médica virtual',
  'puntos': 20,
  'explicaciones': ['P1: Ubicación conveniente (+20)']}]

❌ DESCARTADAS (Por restricciones clínicas/logísticas):
[{'nombre': 'Ginecología - Hospital B',
  'razones': ['R1: Tiempo de espera supera la tolerancia clínica',
              'R2: Requiere pase a especialidad']},
 {'nombre': 'Mastografía preventiva - Clínica C',
  'razones': ['R1: Tiempo de espera supera la tolerancia clínica',
              'R4: Paciente fuera del rango de edad para este protocolo']}]

PACIENTE: 